# L14a: A General Factor Model and Paper Trading
Today, we'll finish building synthetic markets of many stocks. We'll check whether a synthetic market moves together like the real one, and extend the sector factor model with economic factors and pairs of sectors. We will also discuss a key aspect of testing any trading strategy, namely, how to use a paper trading account and a trading API to place trades.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Check a synthetic market of many stocks against the real market:__ Compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data. Check the stylized facts of one synthetic stock against its real growth rates.
> * __Build a synthetic market from a general factor model:__ Fit a general factor model with the market, economic factors, and sector factors. Use the Cholesky decomposition step to draw correlated factors from independent jump hidden Markov models. This old trick allows us to close a key gap!
> * __Place paper trades through a trading API:__ Describe a paper trading account. Send orders for a basket of stocks to a paper account, and follow each order until it fills or we cancel it.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * A general factor model explains each stock's daily growth rate with the market, economic factors, sector factors, and a stock-specific part. The Cholesky step turns the correlated factors into uncorrelated series, so that independent jump hidden Markov models can draw them. Mixing the draws back restores the covariance between the factors.
> * A synthetic market built from the sector factor model reproduces the sector blocks of the real correlation matrix and each stock's market sensitivity. It links stocks in different sectors only through the market.
> * A paper account trades simulated money at real market prices, including the impact of market friction. For example, a limit order may not fill. A trading method must check the status of each order before its next decision.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The least-squares steps of the fitting algorithm.
> * What the columns of $\mathbf{Z}$ represent after the Cholesky step.
> * How the MCP server turns a request in plain English into calls to Alpaca's API.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L14a) lists this lecture's symbols with their meanings and units.

Let's get started!

___

## Examples

__We will work through these examples in lecture:__

> [▶ Build a synthetic market of many stocks](CHEME-5660-L14a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data. We also check the stylized facts of one synthetic stock, with and without jumps, against its real growth rates.

> [▶ Paper trade a basket of stocks](alpaca-mcp/README.md). In this example, we connect Claude Code or Codex to an Alpaca paper account through a Model Context Protocol (MCP) server. We check the account and the quotes of the stocks in the basket, place a limit order for each stock, and follow each order until it fills or we cancel it.

___

## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

The jump hidden Markov model has three parts:

* __States:__ The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.
* __Daily step:__ Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state.
* __Jump episodes:__ Episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses.

Then we extended the single-asset jump hidden Markov model to many stocks that move together. We started from the single index model and added a sector factor:

> __Definition (sector factor model):__ A sector factor model adds one factor for each sector of the [Global Industry Classification Standard (GICS)](https://www.msci.com/our-solutions/indexes/gics). 
> 
> Each sector factor describes how the single index residuals of the stocks in its sector move together. Let's model the daily growth rate of stock $i$ in sector $s$ as:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \overbrace{\underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{u_{i}(t)}_{\text{stock-specific}}}^{\text{single index residual }\varepsilon_{i}(t)}
> $$
>
> Here $g_{m}(t)$ is the growth rate of `SPY`, $f_{s}(t)$ is the growth rate of sector $s$ (with the market part removed), and $u_{i}(t)$ is the growth rate that neither the market nor the sector explains.

The market sensitivity $\beta_{i}$ and the sector loading $\delta_{i}$ set how much stock $i$ moves with the market and with its sector. We built the many-stock model from single-asset jump hidden Markov models: one for the market, one for each sector, and one for each stock.

Let's pick up where we left off, and build a synthetic market of many stocks:

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L14a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data. We also check the stylized facts of one synthetic stock, with and without jumps, against its real growth rates.

__Today:__ We extend the sector factor model with economic factors and pairs of sectors. Then we send orders for a basket of stocks to a paper account.

___


## Building a General Factor Model
The sector factor model of the review explains each stock's growth rate with the market, its sector, and a stock-specific part. We extend it in two ways: we add economic factors, such as interest rates, and we keep the covariance between pairs of sectors, such as utilities and real estate.

> __Definition (general factor model):__ A general factor model explains the daily growth rate of stock $i$ with the market, $K$ economic factors, and the 11 sector factors:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{\ell=1}^{K}\eta_{i\ell}\,x_{\ell}(t)}_{\text{economic factors}} + \underbrace{\sum_{s=1}^{11}\delta_{is}\,f_{s}(t)}_{\text{sectors}} + \underbrace{u_{i}(t)}_{\text{stock-specific}}
> $$
>
> where $x_{\ell}(t)$ is economic factor $\ell$ on day $t$, such as the daily change in the 10-year Treasury yield, and $\eta_{i\ell}$ is the sensitivity of stock $i$ to that factor. The sector loading $\delta_{is}$ measures how much stock $i$ moves with sector $s$.
>
> Each sector factor $f_{s}(t)$ is the average growth rate of sector $s$ with the market and economic parts removed. The sector factor model of the review is the special case with no economic factors, where $\delta_{is}$ is zero for every sector except the stock's own.

### Fitting Algorithm

__Initialize__: Given the daily growth rates $\mathbf{g}_{i}$ of the stocks $i=1,\dots,n$ over $T$ days; the market growth rates $\mathbf{g}_{m}$ and the economic factors $\mathbf{x}_{1},\dots,\mathbf{x}_{K}$ over the same days; and the sector of each stock. Form the $T\times(K+2)$ design matrix $\hat{\mathbf{X}}\gets\left[\mathbf{1},\mathbf{g}_{m},\mathbf{x}_{1},\dots,\mathbf{x}_{K}\right]$, where $\mathbf{1}$ is a column of ones, and the $n\times 11$ sector matrix $\mathbf{B}$, with $B_{is}=1$ if stock $i$ is in sector $s$ and $0$ otherwise.

1. __Market and economic loadings:__ For each stock $i$ __do__: solve $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\,\boldsymbol{\theta}_{i}=\hat{\mathbf{X}}^{\top}\mathbf{g}_{i}$ for $\boldsymbol{\theta}_{i}=(\alpha_{i},\beta_{i},\eta_{i1},\dots,\eta_{iK})$, and compute the residual $\boldsymbol{\varepsilon}_{i}\gets\mathbf{g}_{i}-\hat{\mathbf{X}}\boldsymbol{\theta}_{i}$.
2. __Sector factors:__ For each day $t=1,2,\dots,T$ __do__: solve $\mathbf{B}^{\top}\mathbf{B}\,\mathbf{f}(t)=\mathbf{B}^{\top}\boldsymbol{\varepsilon}(t)$ for $\mathbf{f}(t)=(f_{1}(t),\dots,f_{11}(t))$, where $\boldsymbol{\varepsilon}(t)=(\varepsilon_{1}(t),\dots,\varepsilon_{n}(t))$. Each $f_{s}(t)$ is the average of $\varepsilon_{i}(t)$ over the stocks in sector $s$. Stack the $\mathbf{f}(t)$ as the rows of the $T\times 11$ matrix $\mathbf{F}$.
3. __Sector loadings:__ For each stock $i$ __do__: solve $\mathbf{F}^{\top}\mathbf{F}\,\boldsymbol{\delta}_{i}=\mathbf{F}^{\top}\boldsymbol{\varepsilon}_{i}$ for $\boldsymbol{\delta}_{i}=(\delta_{i1},\dots,\delta_{i,11})$, and compute the stock-specific residual $\mathbf{u}_{i}\gets\boldsymbol{\varepsilon}_{i}-\mathbf{F}\boldsymbol{\delta}_{i}$.
4. __The Cholesky step:__ Form the $T\times(K+12)$ factor matrix $\mathbf{Y}\gets\left[\mathbf{g}_{m},\mathbf{x}_{1},\dots,\mathbf{x}_{K},\mathbf{F}\right]$. Compute its column averages $\boldsymbol{\mu}$ and its covariance matrix $\mathbf{\Sigma}_{Y}$. Factor $\mathbf{\Sigma}_{Y}=\mathbf{A}\mathbf{A}^{\top}$, and set $\mathbf{Z}\gets\left(\mathbf{Y}-\mathbf{1}\boldsymbol{\mu}^{\top}\right)\left(\mathbf{A}^{-1}\right)^{\top}$.
5. __Fit the jump hidden Markov models:__ For each column $\mathbf{z}_{j}$ of $\mathbf{Z}$ __do__: fit a single-asset jump hidden Markov model to $\mathbf{z}_{j}$. For each stock $i$ __do__: fit a single-asset jump hidden Markov model to $\mathbf{u}_{i}$.

__Output__: The parameters $\alpha_{i}$, $\beta_{i}$, $\eta_{i\ell}$, and $\delta_{is}$ of every stock, the column averages $\boldsymbol{\mu}$, the matrix $\mathbf{A}$, and the fitted jump hidden Markov models.

### Sampling Algorithm
The algorithm below draws one synthetic path of every stock over $T_{\text{sim}}$ days.

__Initialize__: Given the outputs of the fitting algorithm; each stock's real starting price $S_{i}(0)$; the time step $\Delta t=1/252$ year (one trading day); and the number of days $T_{\text{sim}}$.

1. __Factor draws:__ For each column $j=1,2,\dots,K+12$ __do__: draw a path of $T_{\text{sim}}$ days from the model of $\mathbf{z}_{j}$, and store the path as column $j$ of $\mathbf{Z}^{\text{sim}}$. Mix the draws back: $\mathbf{Y}^{\text{sim}}\gets\mathbf{1}\boldsymbol{\mu}^{\top}+\mathbf{Z}^{\text{sim}}\mathbf{A}^{\top}$. The columns of $\mathbf{Y}^{\text{sim}}$ hold the paths $g_{m}(t)$, $x_{\ell}(t)$, and $f_{s}(t)$.
2. __Stock-specific draws:__ For each stock $i$ __do__: draw a path $u_{i}(1),\dots,u_{i}(T_{\text{sim}})$ from the model of $\mathbf{u}_{i}$.
3. __Compose:__ For each stock $i$ and each day $t=1,2,\dots,T_{\text{sim}}$ __do__: compose the growth rate $g_{i}(t)\gets\alpha_{i}+\beta_{i}\,g_{m}(t)+\sum_{\ell}\eta_{i\ell}\,x_{\ell}(t)+\sum_{s}\delta_{is}\,f_{s}(t)+u_{i}(t)$, and compound the price, starting from $S_{i}(0)$: $S_{i}(t)\gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The factor paths $g_{m}(t)$, $x_{\ell}(t)$, and $f_{s}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t=1,\dots,T_{\text{sim}}$.

Previously, we estimated the single index model by minimizing the sum of squared residuals. Setting the gradient of this sum to zero gave a linear system, the normal equations. Steps 1 to 3 of the fitting algorithm follow the same pattern:

| Step | Fit over | Minimize | Normal equations |
|---|---|---|---|
| 1. Market and economic loadings | the $T$ days of stock $i$ | $\lVert\mathbf{g}_{i}-\hat{\mathbf{X}}\boldsymbol{\theta}_{i}\rVert_{2}^{2}$ | $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\,\boldsymbol{\theta}_{i}=\hat{\mathbf{X}}^{\top}\mathbf{g}_{i}$ |
| 2. Sector factors | the $n$ stocks on day $t$ | $\lVert\boldsymbol{\varepsilon}(t)-\mathbf{B}\,\mathbf{f}(t)\rVert_{2}^{2}$ | $\mathbf{B}^{\top}\mathbf{B}\,\mathbf{f}(t)=\mathbf{B}^{\top}\boldsymbol{\varepsilon}(t)$ |
| 3. Sector loadings | the $T$ days of stock $i$ | $\lVert\boldsymbol{\varepsilon}_{i}-\mathbf{F}\boldsymbol{\delta}_{i}\rVert_{2}^{2}$ | $\mathbf{F}^{\top}\mathbf{F}\,\boldsymbol{\delta}_{i}=\mathbf{F}^{\top}\boldsymbol{\varepsilon}_{i}$ |

The Cholesky step and the sampling algorithm work as a pair:

* __Untangle:__ Step 4 turns the correlated factors into uncorrelated series, the columns of $\mathbf{Z}$.
* __Model:__ Step 5 fits one jump hidden Markov model to each column of $\mathbf{Z}$. Drawing these models independently keeps the correlations right, because the columns are uncorrelated.
* __Mix back:__ The sampling algorithm mixes the new draws with $\mathbf{A}$. This puts the correlation between the factors back.

To see how step 4 untangles the factors, take one day $t$. The column vector $\mathbf{y}(t)$ holds the values of all $K+12$ factors on that day, and $\mathbf{z}(t)$ holds the uncorrelated series:

$$
\begin{align*}
\mathbf{y}(t) &= \boldsymbol{\mu}+\mathbf{A}\,\mathbf{z}(t) && \text{mix back (sampling algorithm)}\\
\mathbf{z}(t) &= \mathbf{A}^{-1}\left(\mathbf{y}(t)-\boldsymbol{\mu}\right) && \text{untangle (step 4)}\\
\operatorname{Cov}\left(\mathbf{z}\right) &= \mathbf{A}^{-1}\,\mathbf{\Sigma}_{Y}\left(\mathbf{A}^{-1}\right)^{\top} && \text{covariance of } \mathbf{z}(t)\\
&= \mathbf{A}^{-1}\mathbf{A}\mathbf{A}^{\top}\left(\mathbf{A}^{-1}\right)^{\top} = \mathbf{I} && \text{substitute } \mathbf{\Sigma}_{Y}=\mathbf{A}\mathbf{A}^{\top}
\end{align*}
$$

The covariance matrix of $\mathbf{z}(t)$ is the identity matrix $\mathbf{I}$. Each series in $\mathbf{z}(t)$ has variance one, and no two of them are correlated. Each series is the residual of a regression of one factor on the factors before it in $\mathbf{Y}$, scaled to variance one.

Step 4 untangles all $T$ days at once. It writes day $t$ as row $t$ of $\mathbf{Z}$ and stacks the rows:

$$
\begin{align*}
\mathbf{z}(t)^{\top} &= \left(\mathbf{y}(t)-\boldsymbol{\mu}\right)^{\top}\left(\mathbf{A}^{-1}\right)^{\top} && \text{transpose the second line}\\
\mathbf{Z} = \begin{bmatrix}\mathbf{z}(1)^{\top}\\ \vdots\\ \mathbf{z}(T)^{\top}\end{bmatrix} &= \begin{bmatrix}\left(\mathbf{y}(1)-\boldsymbol{\mu}\right)^{\top}\\ \vdots\\ \left(\mathbf{y}(T)-\boldsymbol{\mu}\right)^{\top}\end{bmatrix}\left(\mathbf{A}^{-1}\right)^{\top} && \text{stack the } T \text{ days as rows}\\
&= \left(\mathbf{Y}-\mathbf{1}\boldsymbol{\mu}^{\top}\right)\left(\mathbf{A}^{-1}\right)^{\top} && \text{row } t \text{ of } \mathbf{Y} \text{ is } \mathbf{y}(t)^{\top}
\end{align*}
$$

Let's answer two more questions about the algorithms:

* __Why jump hidden Markov models?__ We need to draw new paths of each column of $\mathbf{Z}$ and each stock-specific residual. Normal draws, as in the multiple-asset GBM model, would have the right variance but thin tails and no volatility clustering. A jump hidden Markov model fit to each series reproduces its fat tails, volatility clustering, and jump episodes.
* __Why no residual scale?__ In the sector factor model, a stock's sector part and stock-specific part were correlated in the data, and the residual scale $\psi_{i}$ added back the variance that independent draws lost. Here, the least-squares fit of step 3 makes the stock-specific residual uncorrelated with every factor. Independent draws then lose no variance.

With these algorithms, you can build a synthetic market from the economic factors of your choice, and test a trading method on many paths of that market.

> [▶ Problem Set 6: Does a synthetic market move together like the real market?](https://github.com/varnerlab/PS6-CHEME-4660-5660-Fall-2026). In this problem set, we build a synthetic market from single-asset jump hidden Markov models and compare how its stocks move together with the real ones. The Standard track fits the market and the sector factors. The Advanced track adds the Cholesky step and replaces the sector factors with four economic factors from FRED, the economic database of the Federal Reserve Bank of St. Louis.

___

## Company Profile: Alpaca
[Alpaca](https://alpaca.markets/about) is a brokerage built for programs that trade through an application programming interface (API). Yoshi Yokokawa and Hitoshi Harada founded the company in 2015, and it became a registered broker-dealer in 2018.

> __What does the firm do?__
>
> * __Trading for programs:__ Alpaca's [Trading API](https://docs.alpaca.markets/docs/trading-api) lets a program check an account, read prices, and send orders. Every user gets a free [paper trading account](https://docs.alpaca.markets/docs/paper-trading), which trades simulated money at real market prices.
> * __Brokerage for other apps:__ Alpaca's Broker API lets other firms build trading apps on top of Alpaca's accounts. In [January 2026](https://alpaca.markets/blog/alpaca-raises-150-million-at-a-1-15b-valuation-to-build-the-global-standard-for-brokerage-infrastructure/), Alpaca raised 150 million USD at a valuation of 1.15 billion USD, and it reported over 9 million brokerage accounts in more than 40 countries.
> * __AI assistants that trade:__ Alpaca maintains an [official MCP server](https://github.com/alpacahq/alpaca-mcp-server), which lets an AI assistant such as Claude check an account and place orders. In [July 2026](https://alpaca.markets/blog/alpaca-raises-135-million-to-scale-agent-first-brokerage-infrastructure-for-tokenized-markets-and-ai-native-financial-services), Alpaca raised a second round of 135 million USD to build brokerage services for AI assistants. It reported that its monthly active API users nearly quadrupled over the previous six months.

__Explore further:__

* __Jobs:__ Alpaca's [hiring page](https://alpaca.markets/hiring) says the company works fully remote, with team members in more than 20 countries. In November 2026, its [job board](https://job-boards.greenhouse.io/alpaca) listed roles in engineering, data science, compliance, and brokerage operations.
* __Video:__ Watch [Backend Operations for a Trade](https://varnerlab.org/watch.html?v=KiWT1qq59Gc) on the firm's YouTube channel. In this 2023 video, Alpaca's VP of Operations explains what happens after a customer places a trade.

__Connection to today's lecture:__ We'll send orders for a basket of stocks to an Alpaca paper account through our own MCP server.

___

## Paper Trading with Alpaca
In a trading app, e.g., Robinhood, __we__ formulate and submit an order using our decision process by tapping a button. However, an automated trading method decides its trades in a program, and the program sends its orders through a connection to the market. 

Today, we'll ask an AI assistant to interact with the market through the Alpaca Application Programming Interface (API):

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L14a-PaperTrading-Schematic.svg" width="1000" style="max-width:100%; height:auto;" alt="A laptop running the AI assistant, Claude Code, shows our request, Buy 1 share of KO with a limit order at the bid, and the assistant asking us to approve the tool call submit_order. An arrow labeled tool call runs to the MCP server, a rack of three units labeled get_latest_quotes, submit_order, and get_order, with an orange key labeled key plus secret. An arrow labeled API request runs to the Alpaca paper account, which holds simulated cash and fills each order against the latest market quotes from IEX. Gray arrows carry the order status and the result back."/>
    </center>
</div>

We'll send our orders to a paper account, which lets us test the trading method before it risks real money:

> __Definition (paper account):__ A paper account trades simulated money at real market prices. Alpaca fills each paper order against the latest quotes, but the order never reaches an exchange. A live account sends the same orders to the market with real money.
>
> Thus, a paper account is a safe way to test a trading method, including the impact of market friction, such as the bid-ask spread and limit orders that do not fill.

The assistant calls the Alpaca API through a __Model Context Protocol (MCP) server__, a program that gives the assistant one tool for each request to Alpaca. The server holds the account's key and secret, which work like a user name and a password. The assistant never sees them.

Previously, we defined market and limit orders. A market order fills right away at the best available price, but we do not know that price in advance. A limit order sets the worst price we accept, but it may not fill.

A trading method reads the status of each order before its next decision, because an unfilled limit order leaves the method holding fewer shares than it planned. Let's place orders for a basket of stocks in a paper account:

> __Example:__
>
> [▶ Paper trade a basket of stocks](alpaca-mcp/README.md). In this example, we connect Claude Code or Codex to an Alpaca paper account through an MCP server. We check the account and the quotes, place limit orders, and follow each order until it fills or we cancel it. It assumes that the market is open and that you have the credentials of an Alpaca paper account.

Every estimate we used today was fit once, on the 2014 to 2024 data, and then held fixed. However, a stock's market sensitivity changes over time. By late 2026, `JPM` moved less with the market than its fixed estimate of 1.11 says:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L14a-EWLS-JPM.svg" width="740" style="max-width:100%; height:auto;" alt="The market sensitivity of JPM over 2025 and 2026. A dashed line marks the fixed estimate from the 2014 to 2024 fit, 1.11. The blue updated estimate, with one update per trading day, starts at 1.11, stays near 1.0 through 2025, and drops to about 0.7 in mid-2026. Late in 2026, JPM moves less with the market than the fixed estimate says."/>
    </center>
</div>

How should a trading method respond when the market sensitivity of a stock drifts?

___

## Summary
We checked a synthetic market of many stocks against the real market, extended the sector factor model to a general factor model with economic factors, and sent orders for a basket of stocks to a paper account.

> __Key Takeaways:__
>
> * __Checking a synthetic market:__ We built a synthetic market of 375 stocks from the sector factor model and compared one synthetic path with the real data. Its correlation matrix showed the same sector blocks as the real one, and its market sensitivities were close to the real ones. Stocks in different sectors were linked only through the market.
> * __A general factor model:__ We extended the sector factor model with economic factors and the covariance between pairs of sectors. We used the Cholesky step to turn the correlated factors into uncorrelated series that independent jump hidden Markov models can draw. Mixing the draws back restored the covariance between the factors.
> * __Paper trading:__ We sent limit orders for a basket of stocks through a trading API to a paper account, which trades simulated money at real market prices. Because a limit order may not fill, we followed each order until it filled or we canceled it.

Every estimate we used was fixed at the end of 2024, while a stock's market sensitivity can drift. A trading method that runs every day has to decide what to do about that.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___